🎯 Goal
Load:

🔹 Base model: meta-llama/Llama-2-7b-hf

🔹 Adapter 1: llama2_dapt_lora/ (DAPT via LoRA)

🔹 Adapter 2: qlora_finetuned_model/ (SFT via QLoRA)

And use them together to evaluate chatbot performance using Technique #3: DAPT ➝ SFT.

LLaMA 2 model trained unsupervised + fine-tuned with QLoRA, giving you Technique #3


🧠 Purpose:
This script evaluates Pipeline P3: DAPT ➝ SFT, where:

You first applied unsupervised DAPT (via LoRA)

Then merged it with QLoRA fine-tuned (supervised SFT)

This allows you to test the model’s response after combined pretraining and fine-tuning, without RAG.

Step 1: Imports & Paths
Loads LLaMA 2 + LoRA support + tokenization and quantization.

Sets paths for:

Base model: "meta-llama/Llama-2-7b-hf"

DAPT adapter (llama2_dapt_lora/)

SFT adapter (qlora_finetuned_model/)

In [1]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel, PeftConfig, get_peft_model

# Paths
base_model_id = "meta-llama/Llama-2-7b-hf"
dapt_adapter_path = "../checkpoints/llama2_dapt_lora/"
sft_adapter_path = "../checkpoints/qlora_finetuned_model/"


Step 2: Load Tokenizer
Loads tokenizer and aligns pad_token to eos_token.

In [2]:
tokenizer = AutoTokenizer.from_pretrained(base_model_id, use_auth_token=True)
tokenizer.pad_token = tokenizer.eos_token


c:\Users\berfi\anaconda3\envs\ml_env\lib\site-packages\transformers\models\auto\tokenization_auto.py:809: FutureWarning: The `use_auth_token` argument is deprecated and will be removed in v5 of Transformers. Please use `token` instead.
  warnings.warn(


Step 3: Load Base Model + Attach DAPT Adapter
Loads LLaMA 2 in 4-bit using BitsAndBytesConfig.

Uses device_map="auto" and low_cpu_mem_usage=True to allow CPU offloading.

Applies DAPT LoRA adapter (llama2_dapt_lora/).

In [3]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

# Load 4-bit base model
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
    use_auth_token=True
)

# Apply unsupervised DAPT adapter
model = PeftModel.from_pretrained(base_model, dapt_adapter_path)


c:\Users\berfi\anaconda3\envs\ml_env\lib\site-packages\transformers\models\auto\auto_factory.py:471: FutureWarning: The `use_auth_token` argument is deprecated and will be removed in v5 of Transformers. Please use `token` instead.
  warnings.warn(


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Step 4: Merge SFT Adapter
Applies SFT (QLoRA) adapter on top of the DAPT-modified model.

This creates a merged model with domain-adaptive and instruction-tuned behavior.

In [4]:
# Apply supervised adapter on top of the DAPT-adapted model
model = PeftModel.from_pretrained(model, sft_adapter_path)
model.eval()


PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): PeftModelForCausalLM(
      (base_model): LoraModel(
        (model): LlamaForCausalLM(
          (model): LlamaModel(
            (embed_tokens): Embedding(32000, 4096)
            (layers): ModuleList(
              (0-31): 32 x LlamaDecoderLayer(
                (self_attn): LlamaSdpaAttention(
                  (q_proj): lora.Linear4bit(
                    (base_layer): Linear4bit(in_features=4096, out_features=4096, bias=False)
                    (lora_dropout): ModuleDict(
                      (default): Dropout(p=0.05, inplace=False)
                    )
                    (lora_A): ModuleDict(
                      (default): Linear(in_features=4096, out_features=8, bias=False)
                    )
                    (lora_B): ModuleDict(
                      (default): Linear(in_features=8, out_features=4096, bias=False)
                    )
                    (lora_embedding_A): ParameterDict()
          

Step 5: Chat Function (ask_dapt_sft_model)
Defines a prompt template.

Uses generate() with temperature sampling to produce a conversational response.



In [5]:
def ask_dapt_sft_model(question):
    prompt = f"""You are a helpful assistant at a university.

Answer the following student question clearly and helpfully.

Question: {question}

Answer:"""

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=True,
            temperature=0.7,
            top_k=50,
            top_p=0.95
        )

    answer = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return answer.split("Answer:")[-1].strip()


Step 6: Run Sample Questions
Evaluates 4 real-world university questions:

Booking rooms

PGR Lounge

IT Services

Academic Writing Support

In [6]:
questions = [
    "Where can I book a study room?",
    "What is the PGR Lounge?",
    "How do I contact IT services?",
    "Can I get help with academic writing?",
]

for q in questions:
    print(f"🧑‍🎓 Question: {q}")
    print("🤖 Answer:", ask_dapt_sft_model(q))
    print("-" * 80)


🧑‍🎓 Question: Where can I book a study room?


c:\Users\berfi\anaconda3\envs\ml_env\lib\site-packages\transformers\models\llama\modeling_llama.py:602: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at C:\cb\pytorch_1000000000000\work\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:555.)
  attn_output = torch.nn.functional.scaled_dot_product_attention(


🤖 Answer: You can book a study room in the library. Please visit the Information Desk for more information.
--------------------------------------------------------------------------------
🧑‍🎓 Question: What is the PGR Lounge?
🤖 Answer: The PGR Lounge is a space where postgraduate research students can meet and network with each other. It's also a great place to find out about opportunities, research projects, and workshops.
--------------------------------------------------------------------------------
🧑‍🎓 Question: How do I contact IT services?
🤖 Answer: There are several ways you can contact IT Services:

You can visit the IT Helpdesk in the main library during opening hours, or email it@bradford.ac.uk.

You can also phone the IT Helpdesk on +44 (0) 1274 236000.

Please be aware that you may experience longer wait times during busy periods, including the start of term and exam periods.

The IT Helpdesk is closed during the following holidays: Christmas Day, Boxing Day, New Year's D

✅ Suggestions

Area	Suggestion
File name	evaluate_dapt_sft_chatbot.ipynb
Save outputs	Store answers to JSON/dict for easier export/comparison
Description	Add cell: # This notebook evaluates Pipeline P3 (DAPT + SFT)
Prompt tuning	Consider standardizing prompt templates across all evaluations
✅ Context in Project
This script directly supports:


Pipeline	Description
P3	✅ DAPT + SFT (no RAG)
It is the key intermediary between basic fine-tuning and the full RAG system.